# Project Prometheus — ML Coding Interview

**Format:** Browser-based IDE (CoderPad). Live coding while talking through decisions.

**What they test:** "Strong fundamentals, good judgment, the ability to connect theory to working systems."
Not memorized APIs — they want you to implement core ideas from scratch and explain *why*.

**Question sources:** alirezadir/Machine-Learning-Interviews · amitshekhariitbhu/ml-interview-questions · dennybritz/reinforcement-learning · cloudxlab backprop series · devinterview.io · exponent deep learning 39 · wecreateproblems RL bank

**Target:** ~6 min per question. Questions are ordered by frequency of appearance in real interviews.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

## Q1 — Implement gradient descent for linear regression from scratch

**Appears in:** alirezadir/Machine-Learning-Interviews (implement list), amitshekhariitbhu ("Implement a simple linear regression model from scratch"), aicourseusa.com ("Implement gradient descent for linear regression"), cloudxlab backprop series.

**When you'd use this in real life:**
Linear regression with gradient descent is the *simplest possible training loop* — one weight matrix, one loss, one backward pass. Every neural network you will ever train is this exact loop, just with more layers. If you understand every line here, you understand 80% of what happens inside `optimizer.step()`. Interviewers use it as a litmus test: can you implement the full forward → loss → backward → update cycle without a framework?

**The math:**
```
prediction:  ŷ = X @ w + b                    (matrix form for a batch)
loss:        L = (1/N) Σ (ŷᵢ - yᵢ)²          (MSE)
gradients:   ∂L/∂w = (2/N) Xᵀ(ŷ - y)
             ∂L/∂b = (2/N) Σ(ŷ - y)
update:      w ← w - α · ∂L/∂w
```

**TASK — implement everything below using only NumPy (no PyTorch, no sklearn):**

```python
X_train  # shape [100, 1], input feature
y_train  # shape [100],    true targets  (true: y = 3x + 2 + noise)
```

1. `predict(X, w, b)` → `X @ w + b`
2. `mse_loss(y_pred, y_true)` → scalar
3. `gradients(X, y_pred, y_true)` → `(dw, db)` using the formulas above
4. Training loop for 200 steps with `lr=0.1`: update `w` and `b` each step, print loss every 50 steps.

Final `w` should be close to `3.0`, `b` close to `2.0`.

**PREDICT before running:**
- What happens if `lr=10`? What if `lr=0.000001`?
- Why divide by N when computing gradients?
- What would change if you switched from MSE to MAE?

In [ ]:
np.random.seed(0)
N = 100
X_train = np.random.randn(N, 1)
y_train = 3.0 * X_train.squeeze() + 2.0 + 0.3 * np.random.randn(N)

# Initialize parameters
w = np.zeros((1, 1))
b = 0.0
lr = 0.1

def predict(X, w, b):
    """X: [N,1], w: [1,1], b: scalar → ŷ: [N]"""
    # YOUR CODE HERE
    pass

def mse_loss(y_pred, y_true):
    """y_pred, y_true: [N] → scalar"""
    # YOUR CODE HERE
    pass

def gradients(X, y_pred, y_true):
    """Returns (dw shape [1,1], db scalar)"""
    # YOUR CODE HERE: use the closed-form formulas above
    pass

# Training loop
for step in range(200):
    y_pred = predict(X_train, w, b)
    loss   = mse_loss(y_pred, y_train)
    dw, db = gradients(X_train, y_pred, y_train)
    w = w - lr * dw
    b = b - lr * db
    if (step + 1) % 50 == 0:
        print(f"step={step+1:3d}  loss={loss:.4f}  w={w[0,0]:.3f}  b={b:.3f}")

print(f"\nFinal: w={w[0,0]:.3f} (true=3.0)  b={b:.3f} (true=2.0)")

## Q2 — Implement sigmoid, tanh, ReLU, and softmax activation functions from scratch

**Appears in:** amitshekhariitbhu ("Implement Sigmoid, Tanh, ReLU, LeakyReLU, and Softmax Activation Functions"), alirezadir ML coding list, andrewekhalel/MLQuestions ("Why is ReLU better and more often used than Sigmoid?"), youssefHosni ("Why is Sigmoid/Tanh not preferred in hidden layers?").

**When you'd use this in real life:**
Every forward pass calls an activation function. The choice matters:
- **Sigmoid** — use only at the output layer for binary classification (not in hidden layers — see vanishing gradient below)
- **Tanh** — zero-centered version of sigmoid; still saturates, still vanishes
- **ReLU** — default for hidden layers; fast, doesn't saturate for positive inputs
- **Softmax** — use only at output layer for multi-class classification; converts logits to a probability distribution

The vanishing gradient problem is one of the most common interview follow-up questions. Be ready to explain it from the derivative.

**TASK — implement in NumPy only:**

1. `sigmoid(x)` — formula: `1 / (1 + e^(-x))`
2. `sigmoid_grad(x)` — formula: `sigmoid(x) * (1 - sigmoid(x))`
3. `tanh_manual(x)` — formula: `(e^x - e^(-x)) / (e^x + e^(-x))`
4. `relu(x)` — formula: `max(0, x)` element-wise
5. `relu_grad(x)` — 1 where x > 0, else 0
6. `softmax(x)` — formula: `e^x / sum(e^x)` (implement the **numerically stable** version here — subtract max first)

Then run the **vanishing gradient demo**: print `sigmoid_grad` at x = 0, 2, 5, 10. Simulate what happens to a gradient as it flows backward through 10 sigmoid layers.

**PREDICT:**
- `sigmoid_grad(10)` ≈ ? (compute by hand — sigmoid(10) ≈ 1.0)
- After 10 layers each multiplying by `sigmoid_grad(5)`, what is the remaining gradient?
- `relu_grad(0)` is technically undefined — what do implementations typically return?

In [ ]:
def sigmoid(x):       # YOUR CODE HERE
    pass
def sigmoid_grad(x):  # YOUR CODE HERE
    pass
def tanh_manual(x):   # YOUR CODE HERE
    pass
def relu(x):          # YOUR CODE HERE
    pass
def relu_grad(x):     # YOUR CODE HERE
    pass
def softmax(x):       # YOUR CODE HERE — numerically stable (subtract max)
    pass

# --- Verify against NumPy/reference ---
x = np.array([-2., -1., 0., 1., 2.])
print("sigmoid:  ", np.round(sigmoid(x), 4), " | ref:", np.round(1/(1+np.exp(-x)), 4))
print("tanh:     ", np.round(tanh_manual(x), 4), " | ref:", np.round(np.tanh(x), 4))
print("relu:     ", relu(x), " | ref:", np.maximum(0, x))
print("relu_grad:", relu_grad(x))
print()
print("softmax([1,2,3]):", np.round(softmax(np.array([1.,2.,3.])), 4), " | sums to:", softmax(np.array([1.,2.,3.])).sum())
print("softmax stable on [1000,1001,1002]:", softmax(np.array([1000.,1001.,1002.])))  # should NOT be nan
print()

# --- Vanishing gradient demo ---
print("=== Vanishing gradient through sigmoid ===")
for xv in [0., 2., 5., 10.]:
    g = sigmoid_grad(np.array(xv))
    print(f"  sigmoid_grad({xv:4.1f}) = {g:.6f}")

print()
print("Gradient surviving 10 sigmoid layers (each saturated at x=5):")
grad = 1.0
for layer in range(1, 11):
    grad *= sigmoid_grad(np.array(5.0))
    print(f"  after layer {layer:2d}: {grad:.2e}")

## Q3 — Implement MSE, binary cross-entropy, and categorical cross-entropy from scratch

**Appears in:** amitshekhariitbhu ("Write a Python function to compute MSE / MAE"), Sroy20 ("Compare L1-loss vs L2-loss"), exponent ("Explain loss and activation functions"), youssefHosni (regularization + loss questions throughout).

**When you'd use this in real life:**
| Task | Loss to use | Why |
|---|---|---|
| Regression | MSE | Penalizes large errors quadratically |
| Binary classification | BCE | Correct probabilistic interpretation for 0/1 labels |
| Multi-class classification | Categorical CE | Correct for mutually exclusive classes |
| RL policy gradient | Log-prob (negative CE) | Maximize log π(a\|s) — this IS cross-entropy |

The most common mistake interviewers look for: using MSE for classification. It technically works but learns the wrong thing. Know *why* CE is correct for classification (it's the maximum likelihood estimator for a categorical distribution).

**TASK — implement in NumPy only:**

1. `mse(y_pred, y_true)` — `mean((ŷ - y)²)`
2. `mae(y_pred, y_true)` — `mean(|ŷ - y|)` (bonus — shows you know both)
3. `binary_cross_entropy(logits, y_true)` — apply sigmoid first, then `-[y·log(p) + (1-y)·log(1-p)]`, clamp `p` to `[1e-7, 1-1e-7]`
4. `cross_entropy(logits, y_true)` — apply softmax first, then `-log(p[correct_class])`, mean over batch

Verify each against a manual calculation you work out first.

**PREDICT:**
- If a binary classifier always outputs `p=0.5`, what is the BCE loss? (compute: `-[1·log(0.5) + 0·log(0.5)]`)
- Why do we pass logits rather than probabilities to cross-entropy functions in practice?
- MSE for a classifier: if true class is 1 and prediction is 0.9, MSE = 0.01. BCE = ? Which punishes more strongly?

In [ ]:
def mse(y_pred, y_true):
    # YOUR CODE HERE
    pass

def mae(y_pred, y_true):
    # YOUR CODE HERE
    pass

def binary_cross_entropy(logits, y_true):
    """logits: [N], y_true: [N] of 0s and 1s → scalar"""
    # YOUR CODE HERE
    # p = sigmoid(logits), clamp, then BCE formula
    pass

def cross_entropy(logits, y_true):
    """logits: [N, C], y_true: [N] int class indices → scalar"""
    # YOUR CODE HERE
    # probs = softmax(logits, axis=-1), then -log(probs[i, y_true[i]]).mean()
    pass


# --- Verify ---
y_pred_reg = np.array([2.5, 0.0, 2.0, 8.0])
y_true_reg = np.array([3.0,-0.5, 2.0, 7.0])
print(f"MSE: {mse(y_pred_reg, y_true_reg):.4f}  (manual: {np.mean((y_pred_reg-y_true_reg)**2):.4f})")
print(f"MAE: {mae(y_pred_reg, y_true_reg):.4f}  (manual: {np.mean(np.abs(y_pred_reg-y_true_reg)):.4f})")
print()

logits_bin = np.array([2.0, -1.0, 0.5, -0.5])
y_true_bin = np.array([1.0,  0.0, 1.0,  0.0])
my_bce = binary_cross_entropy(logits_bin, y_true_bin)
ref_bce = -np.mean(y_true_bin * np.log(sigmoid(logits_bin) + 1e-7) +
                   (1 - y_true_bin) * np.log(1 - sigmoid(logits_bin) + 1e-7))
print(f"BCE: mine={my_bce:.4f}  ref={ref_bce:.4f}")
print()

logits_cat = np.array([[2.,1.,0.1],[0.5,2.5,0.3],[1.,0.5,3.]])
y_true_cat = np.array([0, 1, 2])
my_ce = cross_entropy(logits_cat, y_true_cat)
ref_probs = softmax(logits_cat) if logits_cat.ndim == 1 else np.array([softmax(r) for r in logits_cat])
ref_ce = -np.mean(np.log(ref_probs[np.arange(3), y_true_cat] + 1e-7))
print(f"CE:  mine={my_ce:.4f}  ref={ref_ce:.4f}")
print()
print("Manual check — BCE when p=0.5 for all samples:")
print(f"  Expected: {-np.log(0.5):.4f}  Got: {binary_cross_entropy(np.zeros(4), np.ones(4)):.4f}")

## Q4 — Implement backpropagation through a 2-layer MLP from scratch

**Appears in:** cloudxlab ("Coding Backpropagation and Gradient Descent From Scratch without using any libraries"), alirezadir (implement feedforward NN from scratch), tensorgym ("Multi-layer perceptron implementation with forward and backward passes — a standard ML coding interview task"), sundeepteki.org ("Derive the gradients for this specific custom layer").

**When you'd use this in real life:**
This is the most important question in the entire worksheet. PyTorch's autograd computes these exact gradients automatically, but understanding where they come from lets you: (1) write custom layers for research, (2) debug exploding/vanishing gradients, (3) confidently answer "what does backward() actually compute?" in an interview. The chain rule applied to matrices is the engine of all deep learning.

**Architecture:**
```
Input X [N, D_in]
 → Linear1: Z1 = X @ W1 + b1        [N, H]
 → ReLU:    A1 = relu(Z1)            [N, H]
 → Linear2: Z2 = A1 @ W2 + b2       [N, D_out]
 → Loss:    L = MSE(Z2, y)           scalar
```

**Backward pass (chain rule — derive these before looking):**
```
dL/dZ2 = (2/N)(Z2 - y)          ← MSE gradient
dW2    = A1.T @ dZ2
db2    = dZ2.sum(axis=0)
dA1    = dZ2 @ W2.T
dZ1    = dA1 * relu_grad(Z1)    ← elementwise, chain through ReLU
dW1    = X.T @ dZ1
db1    = dZ1.sum(axis=0)
```

**TASK — NumPy only:**
1. `forward(X, W1, b1, W2, b2)` — returns `(Z2, cache)` where cache stores everything needed for backward
2. `backward(dZ2, cache, W2)` — returns `(dW1, db1, dW2, db2)`
3. Verify: run one forward + backward, then confirm your `dW1` matches PyTorch autograd's result

**PREDICT:**
- Shape of `dW1`? Of `dW2`? (derive from the matrix multiply rules)
- Why does `dZ1 = dA1 * relu_grad(Z1)` use `*` (elementwise) not `@` (matrix multiply)?

In [ ]:
def forward(X, W1, b1, W2, b2):
    """
    2-layer MLP forward pass.
    Returns (Z2: output, cache: dict of intermediates for backward).
    """
    # YOUR CODE HERE
    # Z1 = X @ W1 + b1
    # A1 = relu(Z1)
    # Z2 = A1 @ W2 + b2
    # cache = {'X': X, 'Z1': Z1, 'A1': A1}
    pass

def backward(dZ2, cache, W2):
    """
    2-layer MLP backward pass.
    dZ2: gradient of loss w.r.t. Z2, shape [N, D_out]
    Returns (dW1, db1, dW2, db2).
    """
    # YOUR CODE HERE — use the formulas from the description above
    pass


# --- Set up parameters ---
np.random.seed(42)
N, D_in, H, D_out = 8, 4, 16, 1

W1 = np.random.randn(D_in, H) * 0.1
b1 = np.zeros(H)
W2 = np.random.randn(H, D_out) * 0.1
b2 = np.zeros(D_out)
X  = np.random.randn(N, D_in)
y  = np.random.randn(N, D_out)

# --- Forward + backward (numpy) ---
Z2, cache = forward(X, W1, b1, W2, b2)
N_ = X.shape[0]
dZ2 = (2 / N_) * (Z2 - y)
dW1_np, db1_np, dW2_np, db2_np = backward(dZ2, cache, W2)

# --- Verify against PyTorch autograd ---
W1_t = torch.tensor(W1, requires_grad=True, dtype=torch.float64)
b1_t = torch.tensor(b1, requires_grad=True, dtype=torch.float64)
W2_t = torch.tensor(W2, requires_grad=True, dtype=torch.float64)
b2_t = torch.tensor(b2, requires_grad=True, dtype=torch.float64)
X_t  = torch.tensor(X, dtype=torch.float64)
y_t  = torch.tensor(y, dtype=torch.float64)

A1_t = torch.relu(X_t @ W1_t + b1_t)
Z2_t = A1_t @ W2_t + b2_t
loss_t = ((Z2_t - y_t)**2).mean()
loss_t.backward()

print(f"dW1 match: {np.allclose(dW1_np, W1_t.grad.numpy(), atol=1e-10)}")
print(f"db1 match: {np.allclose(db1_np, b1_t.grad.numpy(), atol=1e-10)}")
print(f"dW2 match: {np.allclose(dW2_np, W2_t.grad.numpy(), atol=1e-10)}")
print(f"db2 match: {np.allclose(db2_np, b2_t.grad.numpy(), atol=1e-10)}")
print(f"\ndW1 shape: {dW1_np.shape}  (expected: {W1.shape})")
print(f"dW2 shape: {dW2_np.shape}  (expected: {W2.shape})")

## Q5 — Implement batch normalization from scratch

**Appears in:** devinterview.io ("Implement batch normalization with NumPy — a common question"), andrewekhalel/MLQuestions ("What is batch normalization and why does it work?"), youssefHosni ("Why should we use Batch Normalization?", "What are the hyperparameters that can be optimized for the batch normalization layer?"). Consistently listed in every serious ML interview prep guide as a must-know implementation.

**When you'd use this in real life:**
Without batch norm, training deep networks (10+ layers) is very unstable — activations can grow or shrink exponentially as they propagate forward. BatchNorm fixes this by normalizing each layer's output to have mean 0 and std 1 *during training*, then learning the right scale (γ) and shift (β) as parameters. It's in almost every modern architecture: ResNets, BERT (as LayerNorm), etc.

**The algorithm:**
```
During training (for a batch of inputs X with shape [N, D]):
  μ  = X.mean(axis=0)               ← mean over batch dimension
  σ² = X.var(axis=0)                ← variance over batch dimension
  X̂  = (X - μ) / sqrt(σ² + ε)      ← normalize
  Y  = γ * X̂ + β                   ← scale and shift (γ, β are learned)

During inference:
  Use running_mean and running_var (accumulated during training), NOT the batch stats.
  X̂ = (X - running_mean) / sqrt(running_var + ε)
  Y  = γ * X̂ + β
```

**TASK:**
Implement `BatchNorm1d` as a plain Python class with:
- `__init__(D, eps=1e-5, momentum=0.1)`: initialize `gamma=ones(D)`, `beta=zeros(D)`, `running_mean=zeros(D)`, `running_var=ones(D)`
- `forward(X, training=True)`: implement both training and inference paths
- Update running stats during training: `running_mean = (1-momentum)*running_mean + momentum*mean`

Verify: `forward(X, training=True)` output should have mean≈0 and std≈1 per feature (before γ/β).

**PREDICT:**
- Why do we need *running* mean/var for inference instead of just using batch stats?
- What does γ=1, β=0 do? What does the network learn γ and β for?
- Why add ε (epsilon) to the variance before taking sqrt?

In [ ]:
class BatchNorm1d:
    def __init__(self, D, eps=1e-5, momentum=0.1):
        self.eps      = eps
        self.momentum = momentum
        self.gamma    = np.ones(D)
        self.beta     = np.zeros(D)
        # Running stats for inference (not updated with gradients)
        self.running_mean = np.zeros(D)
        self.running_var  = np.ones(D)

    def forward(self, X, training=True):
        """
        X: [N, D]  →  Y: [N, D]
        training=True:  normalize with batch stats, update running stats
        training=False: normalize with running stats
        """
        if training:
            # YOUR CODE HERE
            # mu = X.mean(axis=0)
            # var = X.var(axis=0)
            # X_hat = (X - mu) / sqrt(var + eps)
            # update running_mean and running_var
            # return gamma * X_hat + beta
            pass
        else:
            # YOUR CODE HERE — use running_mean and running_var
            pass


# --- Verify ---
np.random.seed(0)
bn = BatchNorm1d(D=4)

# Inputs with non-zero mean and non-unit variance per feature
X_in = np.random.randn(32, 4) * 5 + 10  # mean≈10, std≈5

Y_train = bn.forward(X_in, training=True)
print("After BatchNorm (training=True):")
print(f"  Output mean per feature: {Y_train.mean(axis=0).round(4)}  (expected ≈ 0)")
print(f"  Output std  per feature: {Y_train.std(axis=0).round(4)}   (expected ≈ 1)")
print(f"  Running mean after 1 batch: {bn.running_mean.round(4)}")
print()

# Verify running stats converge with many batches
for _ in range(100):
    X_batch = np.random.randn(32, 4) * 5 + 10
    bn.forward(X_batch, training=True)
print(f"Running mean after 100 batches: {bn.running_mean.round(2)}  (expected ≈ [10,10,10,10])")
print(f"Running var  after 100 batches: {bn.running_var.round(2)}   (expected ≈ [25,25,25,25])")
print()

# Compare to PyTorch
bn_ref = nn.BatchNorm1d(4)
X_t = torch.tensor(X_in, dtype=torch.float32)
Y_ref = bn_ref(X_t).detach().numpy()
print(f"Output matches PyTorch BN (atol=1e-4): {np.allclose(Y_train, Y_ref, atol=1e-4)}")

## Q6 — Implement dropout (forward pass + training vs. inference mode)

**Appears in:** Sroy20/machine-learning-interview-questions ("How will you implement dropout during forward and backward pass?"), youssefHosni ("What is the effect of dropout on the training and prediction speed of your deep learning model?"), exponent deep learning 39. One of the top 15 most-asked ML implementation questions across all sources.

**When you'd use this in real life:**
Dropout prevents overfitting by randomly zeroing neurons during training, forcing the network to learn redundant representations. The critical detail is **inverted dropout**: scale the surviving neurons by `1/(1-p)` during training so the expected value stays the same. Without this, you'd have to scale all outputs by `(1-p)` at inference — which is slower and easy to forget.

**The algorithm:**
```
Training:
  mask = Bernoulli(1 - p)           ← 1 with prob (1-p), 0 with prob p
  out  = x * mask / (1 - p)         ← zero out + rescale (inverted dropout)

Inference (training=False):
  out  = x                           ← no change needed
```

**TASK — implement in NumPy:**

`dropout(X, p, training=True)`:
- Training: sample a mask with `np.random.binomial`, zero out and rescale
- Inference: return X unchanged

Then demonstrate:
1. During training with `p=0.5`, ~50% of values are zeroed
2. The mean of the output is approximately equal to the mean of the input (the rescaling preserves it)
3. During inference, the output is identical to the input

**PREDICT:**
- With `p=0.5`, if input value is 1.0, expected output during training = ?
- Why does dropout act as a regularizer? (think: what ensemble of networks is it approximating?)
- Why is dropout turned off at inference but batch norm's behavior also changes at inference? What do they have in common?

In [ ]:
def dropout(X, p, training=True):
    """
    Inverted dropout.
    X: np.ndarray of any shape
    p: drop probability (fraction of neurons zeroed)
    training: if False, return X unchanged
    """
    if not training:
        return X
    # YOUR CODE HERE
    # mask = np.random.binomial(1, 1-p, size=X.shape)   ← 1 with prob (1-p)
    # return X * mask / (1 - p)
    pass


np.random.seed(0)
X_in = np.ones(10000)   # all ones — easy to track the mean

# --- Check 1: ~50% zeroed ---
out_train = dropout(X_in, p=0.5, training=True)
frac_zero = (out_train == 0).mean()
print(f"p=0.5, fraction zeroed: {frac_zero:.3f}  (expected ~0.5)")

# --- Check 2: mean preserved ---
print(f"Mean of input:  {X_in.mean():.3f}")
print(f"Mean of output: {out_train.mean():.3f}  (should be ≈ 1.0 — same as input)")

# --- Check 3: inference unchanged ---
out_eval = dropout(X_in, p=0.5, training=False)
print(f"Inference unchanged: {np.array_equal(out_eval, X_in)}")
print()

# --- Show effect of different p values ---
print("p   frac_zeroed  mean_output")
for p in [0.1, 0.3, 0.5, 0.7]:
    out = dropout(X_in, p=p, training=True)
    print(f"{p}   {(out==0).mean():.3f}        {out.mean():.3f}")

## Q7 — Build a modular MLP and a complete training loop in PyTorch

**Appears in:** alirezadir ("Implement a feedforward neural network from scratch"), exponent ("What is the purpose of `zero_grad()` in PyTorch?", "Implement a custom training loop"), youssefHosni ("You notice your model is strongly overfitting — what do you do?"). Every single ML coding interview includes this in some form.

**When you'd use this in real life:**
This is the actual job. Every experiment you run will be some variation of this loop. The interviewers specifically said "come ready to write modular, legible code" — this is where they test that. Mistakes they look for: forgetting `zero_grad()`, not using `model.eval()` before evaluation, not using `torch.no_grad()` during eval (wastes memory), hardcoding layer sizes.

**Key things an interviewer checks:**
1. `optimizer.zero_grad()` before `loss.backward()` — every step, always
2. `model.train()` before training, `model.eval()` before eval — affects BatchNorm and Dropout
3. `torch.no_grad()` during eval — no autograd graph needed, saves memory
4. Shuffling data each epoch

**TASK — PyTorch:**
1. `MLP(layer_sizes, activation)` using `nn.ModuleList` — configurable depth, any activation
2. `train_epoch(model, optimizer, X, y)` — mini-batch loop, returns mean loss
3. `evaluate(model, X, y)` — returns accuracy with `model.eval()` + `no_grad()`
4. Train `MLP([2, 64, 64, 2])` for 100 epochs on the synthetic data below. Print loss + val acc every 20 epochs.

**PREDICT:**
- What happens if you forget `zero_grad()`? (gradients accumulate — loss appears to decrease but model diverges)
- What is `nn.ModuleList` and why not use a plain Python list?
- On this easy 2D linearly separable dataset, what val accuracy should you hit?

In [ ]:
class MLP(nn.Module):
    def __init__(self, layer_sizes, activation=nn.ReLU):
        """layer_sizes e.g. [2, 64, 64, 2]: input_dim, *hidden, output_dim"""
        super().__init__()
        # YOUR CODE HERE
        # self.layers = nn.ModuleList(...)
        # self.act    = activation()

    def forward(self, x):
        # YOUR CODE HERE — apply activation after every layer except the last
        pass

def train_epoch(model, optimizer, X, y, batch_size=32):
    model.train()
    # YOUR CODE HERE
    # shuffle → iterate batches → zero_grad → forward → CE loss → backward → step
    # return mean loss
    pass

def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        # YOUR CODE HERE — argmax predictions, return accuracy
        pass


# --- Synthetic 2D data ---
torch.manual_seed(0)
def make_data(n=300):
    half = n // 2
    X0 = torch.randn(half, 2) * 0.6 + torch.tensor([-1.5, -1.5])
    X1 = torch.randn(half, 2) * 0.6 + torch.tensor([ 1.5,  1.5])
    X  = torch.cat([X0, X1])
    y  = torch.cat([torch.zeros(half), torch.ones(half)]).long()
    return X, y

X_all, y_all = make_data()
X_tr, y_tr   = X_all[:240], y_all[:240]
X_val, y_val = X_all[240:], y_all[240:]

# --- Train ---
torch.manual_seed(0)
model = MLP([2, 64, 64, 2])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"{'Epoch':>6}  {'Loss':>8}  {'Val Acc':>8}")
print("-" * 28)
for epoch in range(1, 101):
    loss = train_epoch(model, optimizer, X_tr, y_tr)
    if epoch % 20 == 0:
        acc = evaluate(model, X_val, y_val)
        print(f"{epoch:6d}  {loss:8.4f}  {acc:8.3f}")

## Q8 — Implement scaled dot-product attention from scratch

**Appears in:** alirezadir (implement self-attention / multi-head attention from scratch), tensorgym (described as "a standard ML coding interview task"), yuan-meng.com MLE Interview 2.0 ("implement a Transformer encoder block using only bare PyTorch — no nn.MultiheadAttention"), Amazon Applied Scientist intern 2025 coding round. Increasingly appears at any company building on top of LLMs — which includes this company.

**When you'd use this in real life:**
Attention is the core operation in every transformer: GPT, BERT, T5, every LLM. For a company using LLMs + RL to solve formal math, understanding attention from first principles is directly relevant — you may need to modify it, add constraints, or debug why the model is attending to the wrong tokens.

**The formula:**
```
Attention(Q, K, V) = softmax( Q @ K.T / sqrt(d_k) ) @ V
```
- **Q** (queries): "what am I looking for?"
- **K** (keys): "what do I have?"
- **V** (values): "what do I return if you find me?"
- `sqrt(d_k)` scaling: prevents dot products from growing too large (which would saturate softmax)

For **masked** (causal) attention (used in GPT-style decoders): mask out future positions so position t can only attend to positions ≤ t.

**TASK — PyTorch, no `nn.MultiheadAttention`:**

1. `scaled_dot_product_attention(Q, K, V, mask=None)`:
   - Inputs: Q, K, V each shape `[B, T, d_k]` (batch, sequence length, head dim)
   - Compute scores = `Q @ K.transpose(-2, -1) / sqrt(d_k)`
   - If `mask` provided: add `-1e9` to masked positions (so softmax → 0)
   - Apply softmax over last dim, then `@ V`
   - Return output `[B, T, d_k]` and attention weights `[B, T, T]`

2. Verify: show that with a causal mask, position 0 only attends to position 0, position 1 attends to positions 0-1, etc.

**PREDICT:**
- Why divide by `sqrt(d_k)` and not just `d_k`?
- Without scaling, what happens to softmax as `d_k` grows large?
- What does the attention weight matrix represent geometrically?

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.
    Q, K, V: tensors of shape [B, T, d_k]
    mask:    boolean tensor [B, T, T] or [T, T] — True where we should BLOCK attention
    Returns: (output [B, T, d_k], attn_weights [B, T, T])
    """
    d_k = Q.shape[-1]

    # YOUR CODE HERE
    # scores  = Q @ K.transpose(-2, -1) / sqrt(d_k)     shape: [B, T, T]
    # if mask: scores = scores.masked_fill(mask, -1e9)
    # weights = softmax(scores, dim=-1)
    # output  = weights @ V
    pass


# --- Test 1: basic correctness ---
torch.manual_seed(0)
B, T, d_k = 2, 4, 8
Q = torch.randn(B, T, d_k)
K = torch.randn(B, T, d_k)
V = torch.randn(B, T, d_k)

out, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape:  {out.shape}     (expected: [{B}, {T}, {d_k}])")
print(f"Weights shape: {weights.shape}  (expected: [{B}, {T}, {T}])")
print(f"Weights sum to 1 over last dim: {weights.sum(-1).allclose(torch.ones(B, T))}")
print()

# --- Test 2: causal mask ---
# Upper-triangular mask (block attention to future positions)
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
print("Causal mask (True = blocked):")
print(causal_mask.int().numpy())
print()

out_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print("Attention weights with causal mask (batch 0):")
print(weights_causal[0].detach().numpy().round(3))
print()
print("Each row should only have non-zero weights at or before its position:")
for t in range(T):
    future_weight = weights_causal[0, t, t+1:].sum().item()
    print(f"  position {t}: weight on future positions = {future_weight:.6f}  (expected: 0.0)")

## Q9 — Implement Q-learning (tabular) from scratch

**Appears in:** dennybritz/reinforcement-learning (the canonical RL interview prep repo — exercises on Q-learning, TD, policy gradient), wecreateproblems RL interview bank ("What is Q-learning?", "What is the Bellman equation?"), learnDataSci ("Implement a Q-learning agent from scratch"), interviewnode.com RL interview guide.

**When you'd use this in real life:**
Q-learning is the foundation of all value-based RL. DQN (the algorithm that mastered Atari) is just Q-learning with a neural network as the Q-function. Understanding the tabular version makes you able to explain: (1) why experience replay is needed, (2) why target networks exist, (3) the deadly triad. For this company, even if they use policy gradient methods for LLM training, being able to contrast value-based vs. policy-based approaches from first principles is a core expectation.

**The update rule (from Sutton & Barto — verbatim):**
```
Q(s, a) ← Q(s, a) + α · [ r + γ · max_{a'} Q(s', a') − Q(s, a) ]
                              └─── TD target ───┘   └─ current estimate ─┘
```
The term in brackets is the **TD error**: how wrong our current Q estimate is, given what we just observed.

**TASK — NumPy / pure Python:**

1. `q_update(Q, s, a, r, s_next, done, alpha, gamma)` — one Q-learning update step in-place
2. `epsilon_greedy(Q, state, epsilon)` — random action with prob ε, else argmax
3. Train on the 4-cell GridWorld (state 0→3, reward +1 at goal). 1000 episodes, ε=0.3.
4. Print the converged Q-table. Verify it matches the V* you'd get from value iteration.

**PREDICT:**
- After convergence, `max_a Q(2, a)` should equal `0.9` (one step from goal, γ=0.9). Why?
- What is the difference between Q-learning (off-policy) and SARSA (on-policy)?
- Why does Q-learning converge to the optimal policy even when using ε-greedy (which is not optimal)?

In [ ]:
def gridworld_step(state, action, n=4, goal=3):
    """Deterministic 4-cell GridWorld. Returns (next_state, reward, done)."""
    next_s = np.clip(state + (1 if action == 1 else -1), 0, n - 1)
    reward = 1.0 if next_s == goal else 0.0
    return next_s, reward, next_s == goal

def q_update(Q, s, a, r, s_next, done, alpha=0.1, gamma=0.9):
    """
    One in-place Q-learning update.
    Q: np.ndarray [n_states, n_actions]
    done: bool — if True, no future reward from s_next
    """
    # YOUR CODE HERE
    # td_target = r + gamma * Q[s_next].max() * (not done)
    # td_error  = td_target - Q[s, a]
    # Q[s, a]  += alpha * td_error
    pass

def epsilon_greedy(Q, state, epsilon):
    """Random action with prob epsilon, else argmax Q[state]. Returns int."""
    # YOUR CODE HERE
    pass


# --- Train ---
np.random.seed(0)
Q = np.zeros((4, 2))   # [n_states=4, n_actions=2]  actions: 0=left, 1=right

for ep in range(1000):
    s = 0
    for _ in range(50):
        a = epsilon_greedy(Q, s, epsilon=0.3)
        s_next, r, done = gridworld_step(s, a)
        q_update(Q, s, a, r, s_next, done)
        s = s_next
        if done:
            break

# --- Results ---
print("Converged Q-table:")
print(f"{'State':>6}  {'Q(left)':>8}  {'Q(right)':>9}  {'V*=max_a Q':>11}  {'Best action':>12}")
print("-" * 55)
for s in range(4):
    best = "right" if Q[s,1] > Q[s,0] else "left"
    print(f"{s:6d}  {Q[s,0]:8.4f}  {Q[s,1]:9.4f}  {Q[s].max():11.4f}  {best:>12}")

print()
print("Expected V* by value iteration: [0.729, 0.81, 0.9, 0.0]")
print("   (0.9^3, 0.9^2, 0.9^1 from goal; terminal = 0)")

## Q10 — Implement REINFORCE (policy gradient) from scratch

**Appears in:** dennybritz/reinforcement-learning (REINFORCE exercise — the most-referenced RL coding prep resource), alirezadir ML coding list ("Implement the REINFORCE policy gradient algorithm from scratch using PyTorch"), wecreateproblems RL bank ("What is the REINFORCE algorithm?", "What are baseline functions in policy gradient methods?"). The single most commonly asked RL implementation question.

**When you'd use this in real life:**
REINFORCE is the foundation of everything in RL-for-LLMs: PPO (used in InstructGPT), GRPO (used in DeepSeek-R1), and any RL finetuning of a language model. The core update — `∇θ log π(a|s) · G_t` — is the same equation, just applied to token probabilities instead of simple action probabilities. If you understand this, you understand how ChatGPT was trained.

**The algorithm:**
```
1. Run one episode with policy π_θ, collect (s_t, a_t, r_t)
2. Compute discounted returns: G_t = Σ_{k≥0} γ^k · r_{t+k}
3. Normalize returns: Ĝ_t = (G_t - mean) / std   (variance reduction)
4. Loss = -mean( log π_θ(a_t | s_t) · Ĝ_t )      (negative because we maximize)
5. Backprop and step
```

**TASK — PyTorch:**

Implement a `PolicyNet` and a `reinforce_update` function, then run a full training loop on the GridWorld.

1. `PolicyNet(n_states, n_actions)`: one-hot input → Linear(4→16) → ReLU → Linear(16→2) → logits
2. `select_action(net, state)` → `(action, log_prob)`: one-hot, forward, Categorical sample
3. `compute_returns(rewards, gamma)` → list of G_t (backwards recurrence)
4. `reinforce_update(net, optimizer, log_probs, rewards)`: normalize returns, compute loss, step

Training loop: 600 episodes. Average episode length should drop from ~10 to ~3.

**PREDICT:**
- Why normalize returns? What happens if all returns in an episode are positive?
- The `log_prob` in the loss — this is `log π_θ(a|s)`. Why log and not just prob?
- How is this loss identical in form to maximizing log-likelihood in supervised learning?

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, n_states=4, n_actions=2, hidden=16):
        super().__init__()
        # YOUR CODE HERE — Linear(n_states→hidden) → ReLU → Linear(hidden→n_actions)

    def forward(self, x):
        # YOUR CODE HERE — return logits
        pass

def select_action(net, state_idx, n_states=4):
    """One-hot encode state, run network, sample action. Returns (action int, log_prob tensor)."""
    # YOUR CODE HERE
    # x = one-hot tensor of size n_states
    # logits = net(x)
    # dist = Categorical(logits=logits)
    # action = dist.sample()
    # return action.item(), dist.log_prob(action)
    pass

def compute_returns(rewards, gamma=0.9):
    """G_t = r_t + gamma*r_{t+1} + ... using backwards recurrence. Returns list."""
    # YOUR CODE HERE
    pass

def reinforce_update(net, optimizer, log_probs, rewards, gamma=0.9):
    """Full REINFORCE update: normalize returns, compute loss, backprop, step."""
    returns = torch.tensor(compute_returns(rewards, gamma), dtype=torch.float32)
    # YOUR CODE HERE
    # 1. Normalize: (returns - mean) / (std + 1e-8)
    # 2. loss = -mean(log_prob * return for each t)
    # 3. loss.backward(); optimizer.step(); optimizer.zero_grad()
    pass


# --- Training ---
torch.manual_seed(0)
net = PolicyNet()
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
episode_lengths = []

for ep in range(600):
    s = 0
    log_probs, rewards = [], []
    for _ in range(50):
        a, lp = select_action(net, s)
        s_next, r, done = gridworld_step(s, a)
        log_probs.append(lp)
        rewards.append(r)
        s = s_next
        if done:
            break
    episode_lengths.append(len(rewards))
    reinforce_update(net, optimizer, log_probs, rewards)

    if (ep + 1) % 100 == 0:
        avg = np.mean(episode_lengths[-100:])
        print(f"Episode {ep+1:4d}  avg steps (last 100): {avg:.1f}")

print()
print("Learned policy (greedy argmax):")
with torch.no_grad():
    for s in range(4):
        x = torch.zeros(4); x[s] = 1.0
        logits = net(x)
        action = logits.argmax().item()
        print(f"  state={s}  best_action={'right' if action else 'left'}")